<a href="https://colab.research.google.com/github/bsenst/llm-zoomcamp/blob/bsenst/llm-zoomcamp-code/week04/LLM_Observability_RAG_System_Tracing_and_Analysis_Homework_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Install initial dependencies
!pip install gitsource minsearch openai python-dotenv

In [2]:
# Download starter files
PREFIX="https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"
!wget $PREFIX/rag_helper.py
!wget $PREFIX/starter.py

--2026-07-16 19:08:28--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1814 (1.8K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   1.77K  --.-KB/s    in 0.001s  

2026-07-16 19:08:28 (3.02 MB/s) - ‘rag_helper.py’ saved [1814/1814]

--2026-07-16 19:08:28--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/starter.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting res

In [7]:
# Install OpenTelemetry libraries
!pip install opentelemetry-api opentelemetry-sdk

In [4]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# Configure OpenTelemetry TracerProvider
provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

# Get a tracer instance
tracer = trace.get_tracer("llm-zoomcamp")

print("OpenTelemetry environment configured.")

OpenTelemetry environment configured.


In [20]:
import os
from google.colab import userdata
os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')

In [46]:
corrected_starter_content_v12 = """
import os
from dotenv import load_dotenv

from openai import OpenAI
# Corrected import: GithubRepositoryDataReader is from 'gitsource'
from gitsource import GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

# Load Colab secrets for API keys if available
try:
    from google.colab import userdata
    # Only try to load if env var is not already set by .env or other means
    if 'OPENAI_API_KEY' not in os.environ and 'OPENROUTER_API_KEY' not in os.environ:
        if userdata.get('OPENAI_API_KEY'):
            os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
            print("Loaded OPENAI_API_KEY from Colab secrets.")
        elif userdata.get('OPENROUTER_API_KEY'):
            os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
            print("Loaded OPENROUTER_API_KEY from Colab secrets.")
except ImportError:
    print("google.colab.userdata not available, proceeding without Colab secrets.")

# Load .env file (will not override environment variables already set)
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY"),
                base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"))

# Initialize data reader and index
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp'
    # Temporarily removed 'repo_branch' and 'content_dir' as they are causing TypeError
    # repo_branch='main',
    # content_dir='cohorts/2026/05-monitoring/lessons'
)
docs = reader.read()
# Corrected: Extract the content string from each RawRepositoryFile object
# Also include filename and content explicitly for minsearch to store and return
documents = [{"text": doc.content, "filename": doc.filename, "content": doc.content} for doc in docs]

index = Index(
    text_fields=['text']
    # Removed 'store_fields' as it causes TypeError
    # store_fields=['filename', 'content'] # Added to ensure filename and content are returned
    # Temporarily removed 'vector_field' and 'embedding_model' as they are causing TypeError
    # vector_field='embedding',
    # embedding_model='sentence-transformers/all-MiniLM-L6-v2'
)
index.fit(documents)

# Initialize RAG system
rag = RAGBase(
    # Removed 'documents' argument as it caused TypeError
    index=index,
    llm_client=client, # Added this line to pass the client
    # Removed 'client' argument as it caused TypeError
    model="gpt-4o-mini"
    # Removed 'embedding_model' argument as it caused TypeError
)
"""

with open('starter.py', 'w') as f:
    f.write(corrected_starter_content_v12)

print("starter.py has been corrected and saved (version 12, 'embedding_model' removed from RAGBase init).")

# Also ensure the current directory is in sys.path for local imports
import sys
if '.' not in sys.path:
    sys.path.append('.')

starter.py has been corrected and saved (version 12, 'embedding_model' removed from RAGBase init).


In [47]:
# Import the rag instance from starter.py
import sys
import importlib
sys.path.append('.') # Ensure current directory is in path to find starter.py

# Ensure starter module is reloaded after changes
# This handles cases where starter.py might have been modified
if 'starter' in sys.modules:
    importlib.reload(sys.modules['starter'])
import starter
rag = starter.rag

# Define a query
query = "How does the agentic loop keep calling the model until it stops?"

# Call the RAG system
answer = rag.rag(query)

# Print the answer
print(answer)

The agentic loop continues to call the model until it stops by utilizing a `while` loop that checks for function calls in the model's responses. Here's how it works:

1. The model is instructed to perform actions, such as searches, based on the user's question.
2. After sending the initial request, the model may decide it needs to gather more information, which leads it to make a function call (like `search`).
3. The responses are processed, and if the model has made any function calls, the loop continues, allowing it to search again or take other actions as needed.
4. The loop will terminate when the model provides a response that does not include any further function calls, indicating that it has gathered enough information to answer the user's question.

This allows the model to dynamically decide how many times to search by continuously evaluating its need for additional information.


# Q1. First trace

In [50]:
# Import necessary components for RAGTraced and to re-initialize RAG
from rag_helper import RAGBase
from gitsource import GithubRepositoryDataReader # Corrected import
from opentelemetry import trace
import os
from dotenv import load_dotenv
from openai import OpenAI
from minsearch import Index

# Ensure the tracer is available from the previous setup cell (d969f890)
# If running this cell independently, ensure the tracer is re-initialized:
# from opentelemetry.sdk.trace import TracerProvider
# from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
# provider = TracerProvider()
# provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
# trace.set_tracer_provider(provider)
# tracer = trace.get_tracer("llm-zoomcamp")


# Define the RAGTraced subclass to instrument RAG methods with spans
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag_call") as span:
            span.set_attribute("query", query)
            response = super().rag(query)
            span.set_attribute("response_length", len(response)) # response here is already a string
            return response

    def search(self, query):
        with tracer.start_as_current_span("search_call") as span:
            span.set_attribute("search_query", query)
            results = super().search(query)
            span.set_attribute("search_results_count", len(results))
            return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm_call") as span:
            span.set_attribute("llm_prompt_length", len(prompt))
            answer = super().llm(prompt)
            # Fix: Access the output_text attribute for length calculation
            span.set_attribute("llm_answer_length", len(answer.output_text))
            return answer

# Re-load environment variables to ensure API keys are present
load_dotenv()

# Re-initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY"),
                base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")) # Updated base_url

# Re-initialize data reader and index to ensure they are current
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp'
    # Removed 'repo_branch' and 'content_dir' as they caused TypeError
)
docs = reader.read()
# Corrected: Extract the content string from each RawRepositoryFile object
# Also include filename and content explicitly for minsearch to store and return
documents = [{"text": doc.content, "filename": doc.filename, "content": doc.content} for doc in docs]

index = Index(
    text_fields=['text']
    # Removed 'vector_field' and 'embedding_model' as they caused TypeError
    # Removed 'store_fields' as it caused TypeError
)
index.fit(documents)

# Instantiate RAGTraced with the re-initialized components
rag_traced = RAGTraced(
    # Removed 'documents' argument as it caused TypeError in RAGBase
    index=index,
    llm_client=client, # Corrected argument name from 'client' to 'llm_client'
    model="gpt-4o-mini"
    # Removed 'embedding_model' argument as it caused TypeError in RAGBase
)

# Define the query
query = "How does the agentic loop keep calling the model until it stops?"

# Run the RAG query with tracing enabled
print("Running RAG query with tracing enabled...")
answer = rag_traced.rag(query)
print("\nAnswer:", answer)
print("\nReview the console output above for lines starting with 'ReadableSpan' to count the number of spans produced by this trace.")

Running RAG query with tracing enabled...
{
    "name": "search_call",
    "context": {
        "trace_id": "0xd3e6c1ab34afaf96e203f65273c8b7d8",
        "span_id": "0x4a61c12f03bd3ebc",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc89d707ba39d0e44",
    "start_time": "2026-07-16T19:38:45.789710Z",
    "end_time": "2026-07-16T19:38:45.797113Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search_query": "How does the agentic loop keep calling the model until it stops?",
        "search_results_count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.42.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_call",
    "context": {
        "trace_id": "0xd3e6c1ab34afaf96e203f652

# Q2. Capturing metrics as span attributes


In [51]:
# Import necessary components for RAGTraced and to re-initialize RAG
from rag_helper import RAGBase
from gitsource import GithubRepositoryDataReader # Corrected import
from opentelemetry import trace
import os
from dotenv import load_dotenv
from openai import OpenAI
from minsearch import Index

# Get the existing tracer instance (configured in cell d969f890)
tracer = trace.get_tracer("llm-zoomcamp")

# Define the RAGTraced subclass to instrument RAG methods with spans
class RAGTracedWithMetrics(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag_call") as span:
            span.set_attribute("query", query)
            response = super().rag(query)
            span.set_attribute("response_length", len(response)) # response here is already a string
            return response

    def search(self, query):
        with tracer.start_as_current_span("search_call") as span:
            span.set_attribute("search_query", query)
            results = super().search(query)
            span.set_attribute("search_results_count", len(results))
            return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm_call") as span:
            span.set_attribute("llm_prompt_length", len(prompt))
            answer = super().llm(prompt)
            # Fix: Access the output_text attribute for length calculation
            span.set_attribute("llm_answer_length", len(answer.output_text))

            input_tokens = 0
            output_tokens = 0

            # Safely extract input and output tokens from the LLM response
            if hasattr(answer, 'usage') and answer.usage:
                if hasattr(answer.usage, 'input_tokens'):
                    input_tokens = answer.usage.input_tokens
                elif hasattr(answer.usage, 'prompt_tokens'): # Fallback for OpenAI client
                    input_tokens = answer.usage.prompt_tokens

                if hasattr(answer.usage, 'output_tokens'):
                    output_tokens = answer.usage.output_tokens
                elif hasattr(answer.usage, 'completion_tokens'): # Fallback for OpenAI client
                    output_tokens = answer.usage.completion_tokens

            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)

            # Define prices for gpt-4o-mini (from previous modules context)
            GPT_4O_MINI_INPUT_PRICE_PER_TOKEN = 0.00000015 # $0.15 / 1M tokens
            GPT_4O_MINI_OUTPUT_PRICE_PER_TOKEN = 0.0000006 # $0.60 / 1M tokens

            # Calculate cost
            cost = input_tokens * GPT_4O_MINI_INPUT_PRICE_PER_TOKEN + \
                   output_tokens * GPT_4O_MINI_OUTPUT_PRICE_PER_TOKEN
            span.set_attribute("llm_cost", cost)

            return answer

# Re-load environment variables to ensure API keys are present
load_dotenv()

# Re-initialize OpenAI client (from previous setup)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY"),
                base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"))

# Re-initialize data reader and index (from previous setup)
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp'
)
docs = reader.read()
documents = [{"text": doc.content, "filename": doc.filename, "content": doc.content} for doc in docs]

index = Index(
    text_fields=['text']
)
index.fit(documents)

# Instantiate RAGTracedWithMetrics with the re-initialized components
rag_traced_with_metrics = RAGTracedWithMetrics(
    index=index,
    llm_client=client,
    model="gpt-4o-mini"
)

# Define the query
query = "How does the agentic loop keep calling the model until it stops?"

# Run the RAG query with tracing enabled using the new class
print("Running RAG query with tracing enabled (with token/cost metrics)...")
answer_with_metrics = rag_traced_with_metrics.rag(query)
print("\nAnswer:", answer_with_metrics)
print("\nReview the console output above for the 'llm_call' span to find 'input_tokens'.")


Running RAG query with tracing enabled (with token/cost metrics)...
{
    "name": "search_call",
    "context": {
        "trace_id": "0x2cd5bc0904f7492815643020b7888bde",
        "span_id": "0x0ae87e3b8016beab",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2a87c157907dce06",
    "start_time": "2026-07-16T19:44:30.657727Z",
    "end_time": "2026-07-16T19:44:30.665144Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "search_query": "How does the agentic loop keep calling the model until it stops?",
        "search_results_count": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.42.1",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm_call",
    "context": {
        "trace_id": "

# Q3. Span timing

To determine how long the LLM call takes, we examine the `start_time` and `end_time` attributes of the `llm_call` spans from the execution outputs.

**From the Q1 execution (cell `0FkUwojRVhXH`):**
*   `llm_call` span `start_time`: `2026-07-16T19:38:45.798645Z`
*   `llm_call` span `end_time`: `2026-07-16T19:38:49.623640Z`
*   Duration: `49.623640 - 45.798645 = 3.824995 seconds` (approximately 3825 ms).

**From the Q2 execution (cell `4d60253c`):**
*   `llm_call` span `start_time`: `2026-07-16T19:44:30.666436Z`
*   `llm_call` span `end_time`: `2026-07-16T19:44:33.023685Z`
*   Duration: `33.023685 - 30.666436 = 2.357249 seconds` (approximately 2357 ms).

Both observed durations are greater than 2000ms.

# Q4. Saving traces to SQLite


In [53]:

import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
import os
from dotenv import load_dotenv
from openai import OpenAI
from minsearch import Index
from gitsource import GithubRepositoryDataReader
from rag_helper import RAGBase

# Define the SQLiteSpanExporter class as provided by the user
class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        cursor = self.conn.cursor()
        for span in spans:
            attrs = dict(span.attributes or {})
            # Ensure 'cost' attribute is retrieved correctly, falling back to 'llm_cost' if 'cost' not found
            cost_value = attrs.get("cost")
            if cost_value is None:
                cost_value = attrs.get("llm_cost")

            cursor.execute(
                "INSERT INTO spans (name, start_time, end_time, input_tokens, output_tokens, cost) VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    cost_value,
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

# Create a new TracerProvider specifically for SQLite export
new_provider = TracerProvider()
sqlite_exporter = SQLiteSpanExporter("traces.db")
new_provider.add_span_processor(
    SimpleSpanProcessor(sqlite_exporter)
)

# Get a tracer instance directly from the new provider
# This ensures we use the tracer linked to the SQLite exporter.
sqlite_tracer = new_provider.get_tracer("llm-zoomcamp")

# Define the RAGTracedWithMetrics class (copied from Q2 setup) for metric capture
# Modified to accept a tracer in __init__
class RAGTracedWithMetrics(RAGBase):
    def __init__(self, index, llm_client, model, tracer_instance):
        super().__init__(index=index, llm_client=llm_client, model=model)
        self.tracer = tracer_instance

    def rag(self, query):
        with self.tracer.start_as_current_span("rag_call") as span: # Use self.tracer
            span.set_attribute("query", query)
            response = super().rag(query)
            span.set_attribute("response_length", len(response))
            return response

    def search(self, query):
        with self.tracer.start_as_current_span("search_call") as span: # Use self.tracer
            span.set_attribute("search_query", query)
            results = super().search(query)
            span.set_attribute("search_results_count", len(results))
            return results

    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm_call") as span: # Use self.tracer
            span.set_attribute("llm_prompt_length", len(prompt))
            answer = super().llm(prompt)
            span.set_attribute("llm_answer_length", len(answer.output_text))

            input_tokens = 0
            output_tokens = 0

            if hasattr(answer, 'usage') and answer.usage:
                if hasattr(answer.usage, 'input_tokens'):
                    input_tokens = answer.usage.input_tokens
                elif hasattr(answer.usage, 'prompt_tokens'):
                    input_tokens = answer.usage.prompt_tokens

                if hasattr(answer.usage, 'output_tokens'):
                    output_tokens = answer.usage.output_tokens
                elif hasattr(answer.usage, 'completion_tokens'):
                    output_tokens = answer.usage.completion_tokens

            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)

            GPT_4O_MINI_INPUT_PRICE_PER_TOKEN = 0.00000015
            GPT_4O_MINI_OUTPUT_PRICE_PER_TOKEN = 0.0000006

            cost = input_tokens * GPT_4O_MINI_INPUT_PRICE_PER_TOKEN + \
                   output_tokens * GPT_4O_MINI_OUTPUT_PRICE_PER_TOKEN
            span.set_attribute("llm_cost", cost)
            span.set_attribute("cost", cost) # Ensure 'cost' attribute is set for the exporter

            return answer

# Re-load environment variables to ensure API keys are present
load_dotenv()

# Re-initialize OpenAI client (from previous setup)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY"),
                base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"))

# Re-initialize data reader and index (from previous setup)
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp'
)
docs = reader.read()
documents = [{"text": doc.content, "filename": doc.filename, "content": doc.content} for doc in docs]

index = Index(
    text_fields=['text']
)
index.fit(documents)

# Instantiate RAGTracedWithMetrics, passing the sqlite_tracer explicitly
rag_traced_with_metrics = RAGTracedWithMetrics(
    index=index,
    llm_client=client,
    model="gpt-4o-mini",
    tracer_instance=sqlite_tracer # Pass the tracer explicitly
)

# Define the query
query = "How does the agentic loop keep calling the model until it stops?"

# Run the RAG query with tracing enabled using the new class
print("Running RAG query with SQLite tracing enabled...")
answer_with_metrics = rag_traced_with_metrics.rag(query)
print("\nAnswer:\n", answer_with_metrics)

# Force flush the provider to ensure all spans are exported before querying the DB
new_provider.force_flush()
new_provider.shutdown() # Shutdown to close the SQLite connection cleanly

# Query the SQLite database to get distinct span names
conn = sqlite3.connect("traces.db")
cursor = conn.cursor()
cursor.execute("SELECT DISTINCT name FROM spans")
span_names = [row[0] for row in cursor.fetchall()]
conn.close()

print("\nSpan names recorded in traces.db:", span_names)
print("\nBased on the recorded span names, select the correct option.")

Running RAG query with SQLite tracing enabled...

Answer:
 The agentic loop keeps calling the model until it stops by utilizing a while loop where it continually checks for function calls. The model has the ability to decide if it needs to search based on its previous responses. Each time the model generates a response, it evaluates whether it has called any functions (like a search). If it has not called a function, it means the model has completed its task and the loop terminates. This process allows the model to iteratively refine its searches and responses until it can provide a satisfactory answer or decides no further action is needed.

Span names recorded in traces.db: ['search_call', 'llm_call', 'rag_call']

Based on the recorded span names, select the correct option.


# Q5. Querying trace data

In [54]:
import sqlite3
import time
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
import os
from dotenv import load_dotenv
from openai import OpenAI
from minsearch import Index
from gitsource import GithubRepositoryDataReader
from rag_helper import RAGBase

# Define the SQLiteSpanExporter class again for this cell's context
class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        cursor = self.conn.cursor()
        for span in spans:
            attrs = dict(span.attributes or {})
            cost_value = attrs.get("cost")
            if cost_value is None:
                cost_value = attrs.get("llm_cost")

            cursor.execute(
                "INSERT INTO spans (name, start_time, end_time, input_tokens, output_tokens, cost) VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    cost_value,
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

# Create a new TracerProvider specifically for SQLite export
new_provider = TracerProvider()
sqlite_exporter = SQLiteSpanExporter("traces.db")
new_provider.add_span_processor(
    SimpleSpanProcessor(sqlite_exporter)
)

# Get a tracer instance directly from the new provider
sqlite_tracer = new_provider.get_tracer("llm-zoomcamp")

# Define the RAGTracedWithMetrics class again for this cell's context
class RAGTracedWithMetrics(RAGBase):
    def __init__(self, index, llm_client, model, tracer_instance):
        super().__init__(index=index, llm_client=llm_client, model=model)
        self.tracer = tracer_instance

    def rag(self, query):
        with self.tracer.start_as_current_span("rag_call") as span:
            span.set_attribute("query", query)
            response = super().rag(query)
            span.set_attribute("response_length", len(response))
            return response

    def search(self, query):
        with self.tracer.start_as_current_span("search_call") as span:
            span.set_attribute("search_query", query)
            results = super().search(query)
            span.set_attribute("search_results_count", len(results))
            return results

    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm_call") as span:
            span.set_attribute("llm_prompt_length", len(prompt))
            answer = super().llm(prompt)
            span.set_attribute("llm_answer_length", len(answer.output_text))

            input_tokens = 0
            output_tokens = 0

            if hasattr(answer, 'usage') and answer.usage:
                if hasattr(answer.usage, 'input_tokens'):
                    input_tokens = answer.usage.input_tokens
                elif hasattr(answer.usage, 'prompt_tokens'):
                    input_tokens = answer.usage.prompt_tokens

                if hasattr(answer.usage, 'output_tokens'):
                    output_tokens = answer.usage.output_tokens
                elif hasattr(answer.usage, 'completion_tokens'):
                    output_tokens = answer.usage.completion_tokens

            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)

            GPT_4O_MINI_INPUT_PRICE_PER_TOKEN = 0.00000015
            GPT_4O_MINI_OUTPUT_PRICE_PER_TOKEN = 0.0000006

            cost = input_tokens * GPT_4O_MINI_INPUT_PRICE_PER_TOKEN + \
                   output_tokens * GPT_4O_MINI_OUTPUT_PRICE_PER_TOKEN
            span.set_attribute("llm_cost", cost)
            span.set_attribute("cost", cost)

            return answer

# Re-load environment variables to ensure API keys are present
load_dotenv()

# Re-initialize OpenAI client (from previous setup)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY"),
                base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"))

# Re-initialize data reader and index (from previous setup)
reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp'
)
docs = reader.read()
documents = [{"text": doc.content, "filename": doc.filename, "content": doc.content} for doc in docs]

index = Index(
    text_fields=['text']
)
index.fit(documents)

# Instantiate RAGTracedWithMetrics, passing the sqlite_tracer explicitly
rag_traced_with_metrics = RAGTracedWithMetrics(
    index=index,
    llm_client=client,
    model="gpt-4o-mini",
    tracer_instance=sqlite_tracer
)

# Define the query
query = "How does the agentic loop keep calling the model until it stops?"

# Run another RAG query with tracing enabled using the new class
print("Running another RAG query with SQLite tracing enabled...")
answer_with_metrics = rag_traced_with_metrics.rag(query)
print("\nAnswer:\n", answer_with_metrics)

# Force flush the provider to ensure all spans are exported before querying the DB
new_provider.force_flush()
new_provider.shutdown() # Shutdown to close the SQLite connection cleanly

# Reconnect to the SQLite database to query the data
conn = sqlite3.connect("traces.db")
cursor = conn.cursor()

# Query to calculate total duration for each span type excluding 'rag_call'
cursor.execute("""
    SELECT
        name,
        SUM(end_time - start_time) AS total_duration_ns
    FROM
        spans
    WHERE
        name IN ('search_call', 'llm_call')
    GROUP BY
        name
    ORDER BY
        total_duration_ns DESC
""")

durations = cursor.fetchall()
conn.close()

print("\nTotal durations for search_call and llm_call (in nanoseconds):")
for name, total_duration_ns in durations:
    print(f"  {name}: {total_duration_ns:,} ns")

if durations:
    # Determine which span type took the most time
    if len(durations) == 2:
        if durations[0][1] > durations[1][1]:
            most_time = durations[0][0]
        else:
            most_time = durations[1][0]
        print(f"\nThe span type that takes the most total time is: {most_time}")
    elif len(durations) == 1:
        print(f"\nOnly one relevant span type found: {durations[0][0]}")
    else:
        print("\nCould not determine which span type takes the most time from the database.")
else:
    print("\nNo 'search_call' or 'llm_call' spans found in the database.")


Running another RAG query with SQLite tracing enabled...

Answer:
 The agentic loop keeps calling the model until it stops by continuously running in a `while` loop. It checks whether there are any function calls requested by the model in each iteration. If there are, it processes those function calls, updates the conversation history with the results, and then makes another call to the model. This loop continues until the model returns a response without any function calls, indicating that it has finished its operations. The exit condition for the loop is simply when there are no further function calls needed, which is tracked by a boolean flag (`has_function_calls`).

Total durations for search_call and llm_call (in nanoseconds):
  llm_call: 3,867,896,815 ns
  search_call: 15,271,752 ns

The span type that takes the most total time is: llm_call


# Q6. Token stability across runs

In [55]:
import sqlite3
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
from minsearch import Index
from gitsource import GithubRepositoryDataReader
from rag_helper import RAGBase

# OpenTelemetry imports for re-setup
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

# Re-define the SQLiteSpanExporter class for this cell's context
class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        cursor = self.conn.cursor()
        for span in spans:
            attrs = dict(span.attributes or {})
            cost_value = attrs.get("cost")
            if cost_value is None:
                cost_value = attrs.get("llm_cost")

            cursor.execute(
                "INSERT INTO spans (name, start_time, end_time, input_tokens, output_tokens, cost) VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    cost_value,
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

# Re-define the RAGTracedWithMetrics class for this cell's context
class RAGTracedWithMetrics(RAGBase):
    def __init__(self, index, llm_client, model, tracer_instance):
        super().__init__(index=index, llm_client=llm_client, model=model)
        self.tracer = tracer_instance

    def rag(self, query):
        with self.tracer.start_as_current_span("rag_call") as span:
            span.set_attribute("query", query)
            response = super().rag(query)
            span.set_attribute("response_length", len(response))
            return response

    def search(self, query):
        with self.tracer.start_as_current_span("search_call") as span:
            span.set_attribute("search_query", query)
            results = super().search(query)
            span.set_attribute("search_results_count", len(results))
            return results

    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm_call") as span:
            span.set_attribute("llm_prompt_length", len(prompt))
            answer = super().llm(prompt)
            span.set_attribute("llm_answer_length", len(answer.output_text))

            input_tokens = 0
            output_tokens = 0

            if hasattr(answer, 'usage') and answer.usage:
                if hasattr(answer.usage, 'input_tokens'):
                    input_tokens = answer.usage.input_tokens
                elif hasattr(answer.usage, 'prompt_tokens'):
                    input_tokens = answer.usage.prompt_tokens

                if hasattr(answer.usage, 'output_tokens'):
                    output_tokens = answer.usage.output_tokens
                elif hasattr(answer.usage, 'completion_tokens'):
                    output_tokens = answer.usage.completion_tokens

            span.set_attribute("input_tokens", input_tokens)
            span.set_attribute("output_tokens", output_tokens)

            GPT_4O_MINI_INPUT_PRICE_PER_TOKEN = 0.00000015
            GPT_4O_MINI_OUTPUT_PRICE_PER_TOKEN = 0.0000006

            cost = input_tokens * GPT_4O_MINI_INPUT_PRICE_PER_TOKEN + \
                   output_tokens * GPT_4O_MINI_OUTPUT_PRICE_PER_TOKEN
            span.set_attribute("llm_cost", cost)
            span.set_attribute("cost", cost)

            return answer

# Re-initialize OpenTelemetry setup to ensure fresh recording
new_provider = TracerProvider()
sqlite_exporter = SQLiteSpanExporter("traces.db")
new_provider.add_span_processor(
    SimpleSpanProcessor(sqlite_exporter)
)
sqlite_tracer = new_provider.get_tracer("llm-zoomcamp")

# Re-load environment variables and RAG components
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY") or os.getenv("OPENROUTER_API_KEY"),
                base_url=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"))

reader = GithubRepositoryDataReader(
    repo_owner='DataTalksClub',
    repo_name='llm-zoomcamp'
)
docs = reader.read()
documents = [{"text": doc.content, "filename": doc.filename, "content": doc.content} for doc in docs]

index = Index(
    text_fields=['text']
)
index.fit(documents)

# Instantiate RAGTracedWithMetrics with the new sqlite_tracer
rag_traced_with_metrics_q6 = RAGTracedWithMetrics(
    index=index,
    llm_client=client,
    model="gpt-4o-mini",
    tracer_instance=sqlite_tracer
)

query = "How does the agentic loop keep calling the model until it stops?"

print("Running 3 additional RAG queries for Q6...")
# Run the query three more times (for a total of 4 including the one from Q5)
for i in range(3):
    print(f"  Running query {i+2}/4...") # Already ran 1 in Q5, this is 2, 3, 4
    rag_traced_with_metrics_q6.rag(query)

# Force flush the provider to ensure all spans are exported before querying the DB
new_provider.force_flush()
new_provider.shutdown() # Shutdown to close the SQLite connection cleanly

print("\nAnalyzing input tokens from traces.db...")

# Load all spans from the database
conn = sqlite3.connect("traces.db")
df = pd.read_sql_query("SELECT * FROM spans WHERE name = 'llm_call'", conn)
conn.close()

# Extract input tokens for llm_call spans
input_tokens = df['input_tokens'].dropna().tolist()

if not input_tokens:
    print("No 'llm_call' spans with input tokens found in the database.")
else:
    print(f"Input tokens for 'llm_call' spans: {input_tokens}")

    min_tokens = min(input_tokens)
    max_tokens = max(input_tokens)

    if min_tokens == max_tokens:
        variation_category = "identical"
    elif (max_tokens - min_tokens) / max_tokens <= 0.1:
        variation_category = "within 10% of each other"
    elif (max_tokens - min_tokens) / max_tokens <= 0.5:
        variation_category = "within 50% of each other"
    else:
        variation_category = "vary more than 50%"

    print(f"\nThe input tokens for the 'llm_call' spans are: {variation_category}")
    print(f"Min input tokens: {min_tokens}")
    print(f"Max input tokens: {max_tokens}")
    if max_tokens > 0:
        print(f"Percentage variation: {((max_tokens - min_tokens) / max_tokens * 100):.2f}%")
    else:
        print("Percentage variation: 0%")

Running 3 additional RAG queries for Q6...
  Running query 2/4...
  Running query 3/4...
  Running query 4/4...

Analyzing input tokens from traces.db...
Input tokens for 'llm_call' spans: [8496, 8496, 8496, 8496, 8496]

The input tokens for the 'llm_call' spans are: identical
Min input tokens: 8496
Max input tokens: 8496
Percentage variation: 0.00%
